# DubbingStory — Kaggle Local Vision (Qwen3-VL)

Panduan ini menjelaskan cara menjalankan pipeline DubbingStory di cloud GPU (Kaggle) menggunakan model vision HuggingFace secara lokal, sehingga Anda bisa menganalisa video **tanpa biaya API model vision**!

Kita akan menggunakan **Qwen3-VL-2B-Instruct** (via `vLLM`), model vision open-source yang sangat kompeten dan muat di GPU T4 gratisan yang disediakan Kaggle.

### Persiapan Environment
Pastikan Anda telah mengaktifkan **GPU Accelerator** (T4 x2) di kanan menu pengaturan Kaggle Notebook.

## 1. Clone Repo & Install Dependencies

In [ ]:
# Clone repository langsung ke current directory agar tidak nested
!rm -rf ./* ./.*
!git clone -b main https://github.com/NaufalRizqullah/dubbingstory.git .

# Install dependencies via requirements.txt
# (tambahkan openai dan vllm karena dibutuhkan untuk Local Vision Server)
!pip install -r requirements.txt openai vllm

## 2. Setup API Key via Kaggle Secrets

DubbingStory menggunakan Gemini API **hanya untuk merangkai narasi (script writer)**. Model *vision* akan dijalankan secara lokal.

1. Buka tab **Secrets** (kunci) di panel kiri Kaggle.
2. Tambahkan rahasia baru dengan nama `GOOGLE_API_KEY` dan isikan API Key Gemini Anda.

In [ ]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path

API_KEY_GEMINI = UserSecretsClient().get_secret('GOOGLE_API_KEY') or ""

# Create .env File
env_text = f"""# Auto-generated from notebook userdata
GOOGLE_API_KEY={API_KEY_GEMINI}
"""

Path(".env").write_text(env_text, encoding="utf-8")
print("File .env berhasil dibuat")

## 3. Konfigurasi Pipeline

Pilih mode pipeline:
- `full` → Dubbing seluruh video (default)
- `summary` → Buat highlight recap (video ringkasan dari scene terpenting)

Untuk mode `summary`, Anda bisa mengatur:
- `SUMMARY_DURATION` → Target durasi ringkasan (detik). Set `None` untuk otomatis (~60-120s)
- `SUMMARY_MAX_SCENES` → Maksimum scene yang dipilih. Set `None` untuk otomatis

In [ ]:
# --- SETTINGS ---
VIDEO_INPUT = "https://www.youtube.com/watch?v=7glhmGv9mHk"  # Bisa URL atau path file lokal
PROJECT_NAME = "my_dubbing_project"
STYLE = "viral_fb"
LANGUAGE = "id"
RATIO = "16:9"

# --- PIPELINE MODE ---
MODE = "summary"  # "full" atau "summary"
SUMMARY_DURATION = None  # Target durasi ringkasan (detik), None = otomatis
SUMMARY_MAX_SCENES = None  # Maks scene, None = otomatis

# --- SPEED OPTIMIZATION (Mencegah vLLM hang & mempercepat proses) ---
MAX_KEYFRAMES = 3           # Default 7. Turunkan ke 3-4 agar memori vLLM tidak penuh.
MIN_SCENE_DURATION = 5.0    # Default 2.0. Naikkan ke 5.0 agar scene lebih sedikit.
SCENE_THRESHOLD = 5.0       # Default 3.0. Naikkan ke 5.0 agar tidak over-sensitive.

# --- VISION MODEL ---
MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"
PORT = 8000
BASE_URL = f"http://127.0.0.1:{PORT}/v1"

# --- COLAB T4 vs KAGGLE T4x2 AUTO-DETECT ---
# Colab biasanya kasih 1 GPU (T4 16GB) → max-model-len kecil, tensor-parallel-size 1
# Kaggle T4x2 punya 2 GPU (masing-masing 16GB) → bisa lebih panjang
try:
    import torch
    NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
except Exception:
    NUM_GPUS = 0

if NUM_GPUS >= 2:
    TENSOR_PARALLEL_SIZE = "2"
    MAX_MODEL_LEN = "12288"   # muat di 2x T4 16GB
    GPU_MEM_UTIL = "0.85"
    print(f"🖥️  Detected {NUM_GPUS} GPUs → Kaggle-style: tensor-parallel-size=2, max-model-len=12288")
elif NUM_GPUS == 1:
    TENSOR_PARALLEL_SIZE = "1"
    MAX_MODEL_LEN = "6144"    # konservatif untuk T4 16GB
    GPU_MEM_UTIL = "0.80"
    print(f"🖥️  Detected 1 GPU (Colab T4) → tensor-parallel-size=1, max-model-len=6144")
else:
    TENSOR_PARALLEL_SIZE = "1"
    MAX_MODEL_LEN = "4096"
    GPU_MEM_UTIL = "0.80"
    print("⚠️  No GPU detected — pipeline akan gagal. Pastikan Runtime → Change runtime type → GPU.")

## 4a. Fix CUDA/torchaudio mismatch (Colab only)

Jalankan cell ini **sebelum** cell pipeline di bawah. Colab terbaru (Agustus 2026+) default-nya PyTorch CUDA 13.0, tapi `torchaudio` dari PyPI masih CUDA 12.8 — ini bikin vLLM crash saat import `transformers.loss_ross` (RNN-T loss).

Solusi: hapus `torchaudio` (vLLM tidak butuh), sinkronkan torch+torchvision ke CUDA 13.0, dan upgrade vLLM ke versi yang punya wheel CUDA 13.0.

In [ ]:
import subprocess, sys

print("🔧 Fixing Colab CUDA/torchaudio mismatch...")
print("   - Uninstall torchaudio (vLLM tidak butuh; import-nya cuma lewat transformers.loss_rnnt)")
subprocess.check_call(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)

print("   - Reinstall torch + torchvision sinkron CUDA 13.0 (cocok dengan default Colab)")
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
        "torch==2.13.0", "torchvision==0.28.0",
        "--index-url", "https://download.pytorch.org/whl/cu130",
    ],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)

print("   - Upgrade vLLM ke versi yang punya wheel CUDA 13.0")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "vllm"],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)

print("✅ Done. Lanjut ke cell pipeline di bawah.")

In [ ]:
import subprocess
import time
import urllib.request
import json
import sys
import os
from dotenv import load_dotenv
load_dotenv()

def wait_for_server(url, timeout=600):
    print(f"\n⏳ Waiting for vLLM server to start at {url} (timeout: {timeout}s)...")
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            req = urllib.request.Request(f"{url}/models")
            with urllib.request.urlopen(req) as response:
                if response.status == 200:
                    data = json.loads(response.read().decode())
                    print(f"\n✅ vLLM Server is ready! Available models: {[m['id'] for m in data['data']]}")
                    return True
        except Exception:
            pass
        sys.stdout.write(".")
        sys.stdout.flush()
        time.sleep(5)
    print("\n❌ Timeout waiting for server.")
    return False

if not os.environ.get("GOOGLE_API_KEY"):
    print("❌ ERROR: GOOGLE_API_KEY belum di-set di cell sebelumnya (atau di Kaggle Secrets)!")
else:
    print(f"🚀 Starting vLLM server with model: {MODEL_NAME}...")
    vllm_cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL_NAME,
        "--port", str(PORT),
        # Setting dari cell konfigurasi (Colab 1 GPU vs Kaggle 2 GPU auto-detect)
        "--max-model-len", MAX_MODEL_LEN,
        "--tensor-parallel-size", TENSOR_PARALLEL_SIZE,
        "--gpu-memory-utilization", GPU_MEM_UTIL,
        "--dtype", "bfloat16",
        "--enforce-eager",  # wajib di T4: tidak ada cukup memori untuk CUDA graphs
    ]

    print(f"   Command: {' '.join(vllm_cmd)}")

    vllm_log = open("vllm_server.log", "w")
    vllm_process = subprocess.Popen(vllm_cmd, stdout=vllm_log, stderr=subprocess.STDOUT)

    try:
        if wait_for_server(BASE_URL):
            print(f"\n🚀 Starting DubbingStory Pipeline (mode: {MODE})...")

            cmd = [
                sys.executable, "-u", "main.py", "run",
                "--input", VIDEO_INPUT,
                "--project", PROJECT_NAME,
                "--style", STYLE,
                "--lang", LANGUAGE,
                "--ratio", RATIO,
                "--mode", MODE,
                "--vision-provider", "openai",
                "--vision-model", MODEL_NAME,
                "--vision-base-url", BASE_URL,
                "--engine", "piper",
                "--max-keyframes", str(MAX_KEYFRAMES),
                "--min-scene-duration", str(MIN_SCENE_DURATION),
                "--scene-threshold", str(SCENE_THRESHOLD)
            ]

            # Add summary options if in summary mode
            if MODE == "summary":
                if SUMMARY_DURATION is not None:
                    cmd.extend(["--summary-duration", str(SUMMARY_DURATION)])
                if SUMMARY_MAX_SCENES is not None:
                    cmd.extend(["--summary-max-scenes", str(SUMMARY_MAX_SCENES)])

            if "http" in VIDEO_INPUT:
                cmd[cmd.index("--input")] = "--url"
                cmd.append("--i-have-rights")

            print(f"Executing: {' '.join(cmd)}")
            subprocess.run(cmd, check=True)
            print(f"\n🎉 Pipeline completed successfully! (mode: {MODE})")
            print(f"📂 Check the 'outputs/{PROJECT_NAME}/' directory for your video.")
        else:
            print("Failed to start vision server. Check vllm_server.log for details.")
    except Exception as e:
        print(f"\n❌ An error occurred: {e}")
    finally:
        print("\n🛑 Shutting down vLLM server...")
        vllm_process.terminate()
        try:
            vllm_process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            vllm_process.kill()
            vllm_process.wait()
        vllm_log.close()